# Prompt Engineering Best Practices: Building Chatbots

Maintained runnable companion. Set OPENAI_API_KEY privately in the environment or an uncommitted .env file; never put keys in cells. All model calls incur charges. Originals are retained in the nonexecuting appendix. The notebook is output-free in Git.

In [ ]:
import os, json, re, sys
from importlib.metadata import version
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=60)
def get_completion_from_messages(messages, model=MODEL, temperature=None):
    options = {} if temperature is None else {"temperature": temperature}
    result = client.chat.completions.create(
        model=model, messages=messages, reasoning_effort="none",
        max_completion_tokens=2048, **options)
    if result.choices[0].finish_reason != "stop":
        raise RuntimeError("The completion did not finish normally")
    content = result.choices[0].message.content
    if not content:
        raise RuntimeError("No text returned")
    return content
def get_completion(prompt, model=MODEL, temperature=None):
    return get_completion_from_messages(
        [{"role": "user", "content": prompt}], model=model, temperature=temperature)
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "python-dotenv:", version("python-dotenv"))
print("Model:", MODEL)

In [ ]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [ ]:
response = get_completion_from_messages(messages, temperature=1)
print(response)
joke_response = response

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Youssef'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)
greeting_response = response

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)
without_history_response = response

In [ ]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Youssef'},
{'role':'assistant', 'content': "Hi Youssef! It's nice to meet you.  Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)
with_history_response = response

In [ ]:
import html
from decimal import Decimal
import panel as pn
pn.extension()
MENU = {
    "pepperoni pizza": {"large": "12.95", "medium": "10.00", "small": "7.00"},
    "cheese pizza": {"large": "10.95", "medium": "9.25", "small": "6.50"},
    "eggplant pizza": {"large": "11.95", "medium": "9.75", "small": "6.75"},
    "fries": {"large": "4.50", "regular": "3.50"},
    "greek salad": {"regular": "7.25"},
    "extra cheese": {"regular": "2.00"}, "mushrooms": {"regular": "1.50"},
    "sausage": {"regular": "3.00"}, "canadian bacon": {"regular": "3.50"},
    "AI sauce": {"regular": "1.50"}, "peppers": {"regular": "1.00"},
    "coke": {"large": "3.00", "medium": "2.00", "small": "1.00"},
    "sprite": {"large": "3.00", "medium": "2.00", "small": "1.00"},
    "bottled water": {"regular": "5.00"}}
ORDER_CONTRACT = """Return only a JSON object with exactly two keys: reply and items.
reply: a brief friendly string to collect/clarify the order. Do not include
prices, currency symbols or totals in reply; Python renders all prices.
items: the complete currently confirmed order, an array of objects with exactly
item, size and quantity. Copy item names and sizes from the supplied menu keys.
quantity must be an integer from 1 to 20. Do not include unconfirmed items or
choose an unspecified size; ask the customer to clarify instead. Retain previous
confirmed items unless changed. Do not add Markdown fences or extra keys.
Read explicit choices before asking questions: "one regular fries" confirms
item="fries", size="regular", quantity=1. "medium cheese pizza, no toppings"
confirms the medium pizza and needs no toppings question. Never ask again for
a size already supplied in the current message or conversation. Include every
explicitly ordered item, including fries and water, in items immediately.
"""
def render_order_response(payload):
    if not isinstance(payload, dict) or set(payload) != {"reply", "items"}:
        raise ValueError("Invalid order response fields")
    reply, items = payload["reply"], payload["items"]
    if not isinstance(reply, str) or not reply.strip() or not isinstance(items, list):
        raise ValueError("Invalid reply or items")
    if re.search(r"[$€£]|\d+\.\d{2}\b", reply):
        raise ValueError("Prices must come from Python, not generated reply text")
    total = Decimal("0.00")
    lines = []
    for row in items:
        if not isinstance(row, dict) or set(row) != {"item", "size", "quantity"}:
            raise ValueError("Invalid item fields")
        item, size, quantity = row["item"], row["size"], row["quantity"]
        if not isinstance(item, str) or not isinstance(size, str) or item not in MENU or size not in MENU[item]:
            raise ValueError("Unknown menu item or size")
        if type(quantity) is not int or not 1 <= quantity <= 20:
            raise ValueError("Invalid quantity")
        unit = Decimal(MENU[item][size])
        line_total = unit * quantity
        total += line_total
        lines.append(f"{quantity} × {size} {item}: ${unit:.2f} each = ${line_total:.2f}")
    receipt = "\n\nOrder so far:\n" + "\n".join(lines) + f"\nTotal: ${total:.2f}" if lines else ""
    return reply.strip() + receipt, total
def chat_turn(state, prompt):
    prompt = prompt.strip()
    if not prompt:
        return None
    pending = state + [{"role": "user", "content": prompt}]
    request = pending + [{"role": "system", "content": ORDER_CONTRACT + "\nMenu keys and unit prices: " + json.dumps(MENU)}]
    payload = json.loads(get_completion_from_messages(request))
    response, total = render_order_response(payload)
    state.extend([pending[-1], {"role": "assistant", "content": response}])
    return response
def build_pizzabot(system_message):
    context = [{"role": "system", "content": system_message}]
    inp = pn.widgets.TextInput(label="Message", placeholder="Enter text here…")
    button = pn.widgets.Button(label="Chat!")
    transcript = pn.Column(sizing_mode="stretch_width")
    status = pn.pane.Str("")
    def collect_messages(event=None):
        prompt = inp.value_input.strip()
        if not prompt or button.disabled:
            return
        button.disabled = True
        status.object = ""
        try:
            response = chat_turn(context, prompt)
            transcript.extend([
                pn.pane.HTML("<b>User:</b><pre>" + html.escape(prompt) + "</pre>"),
                pn.pane.HTML("<b>Assistant:</b><pre>" + html.escape(response) + "</pre>",
                             styles={"background-color": "#F6F6F6"})])
            inp.value = ""
            inp.value_input = ""
        except Exception:
            status.object = "Unable to get a reply. Please try again."
        finally:
            button.disabled = False
    button.on_click(collect_messages)
    dashboard = pn.Column(inp, button, status, transcript, max_width=700, sizing_mode="stretch_width")
    return dashboard, context, inp, button, collect_messages
print("Panel:", pn.__version__)
print("PizzaBot callback ready; no API request is made while constructing the UI.")

In [ ]:
pizzabot_system = "You are OrderBot, an automated service to collect orders for a pizza restaurant.  You first greet the customer, then collects the order,  and then asks if it's a pickup or delivery.  You wait to collect the entire order, then summarize it and check for a final  time if the customer wants to add anything else.  If it's a delivery, you ask for an address.  Finally ask only for a payment preference (cash or card); never collect card numbers, security codes or other payment credentials. This is a fictional order demonstration; no order is placed and no payment is processed. Make sure to clarify all options, extras and sizes to uniquely  identify the item from the menu. You respond in a short, very conversational friendly style.  For three listed prices, sizes are large, medium, small in that order. For fries the two prices are large and regular. Drinks with three prices are large, medium, small. Bottled water has one size. Do not invent fees or menu items.\nThe menu includes  pepperoni pizza  12.95, 10.00, 7.00  cheese pizza   10.95, 9.25, 6.50  eggplant pizza   11.95, 9.75, 6.75  fries 4.50, 3.50  greek salad 7.25  Toppings:  extra cheese 2.00,  mushrooms 1.50  sausage 3.00  canadian bacon 3.50  AI sauce 1.50  peppers 1.00  Drinks:  coke 3.00, 2.00, 1.00  sprite 3.00, 2.00, 1.00  bottled water 5.00  "
dashboard, context, inp, button_conversation, collect_messages = build_pizzabot(pizzabot_system)
dashboard.servable(title="PizzaBot")
print("PizzaBot UI ready. Enter a fictional order and click Chat! to call the model.")

In [ ]:
inp.value_input = 'Hi, I would like to order a pizza.'
button_conversation.clicks += 1
assert context[-2]["content"] == 'Hi, I would like to order a pizza.'
print(context[-1]["content"])

In [ ]:
inp.value_input = 'I want one medium cheese pizza, no toppings.'
button_conversation.clicks += 1
assert context[-2]["content"] == 'I want one medium cheese pizza, no toppings.'
print(context[-1]["content"])

In [ ]:
inp.value_input = 'Yes, add one bottled water and one regular fries.'
button_conversation.clicks += 1
assert context[-2]["content"] == 'Yes, add one bottled water and one regular fries.'
print(context[-1]["content"])

In [ ]:
inp.value_input = 'Nothing else. Pickup in one hour; I will pay cash on pickup.'
button_conversation.clicks += 1
assert context[-2]["content"] == 'Nothing else. Pickup in one hour; I will pay cash on pickup.'
print(context[-1]["content"])

In [ ]:
messages = context.copy()
messages.append({"role": "system", "content": "Return only a JSON object summarizing the previous food order. Required keys: pizza (object with type, size, price), toppings (array), drinks (array of objects with type, size, price), sides (array of objects with type, size, price), total_price (number). Prices must match the supplied menu. No Markdown fences or extra keys."})
response = get_completion_from_messages(messages, temperature=0)
print(response)
order_summary = json.loads(response)

In [ ]:
from decimal import Decimal
from bokeh.document import Document
assert joke_response.strip()
assert "youssef" in greeting_response.lower()
assert "youssef" not in without_history_response.lower()
assert "youssef" in with_history_response.lower()
assert len(context) == 9
assert [m["role"] for m in context] == ["system"] + ["user", "assistant"] * 4
assert len(context) == len(messages) - 1
assert order_summary["pizza"]["type"].lower() in {"cheese", "cheese pizza"}
assert order_summary["pizza"]["size"].lower() == "medium"
assert Decimal(str(order_summary["pizza"]["price"])) == Decimal("9.25")
assert order_summary["toppings"] == []
assert len(order_summary["drinks"]) == len(order_summary["sides"]) == 1
assert "water" in order_summary["drinks"][0]["type"].lower()
assert "fries" in order_summary["sides"][0]["type"].lower()
prices = [Decimal(str(order_summary["pizza"]["price"]))] + [Decimal(str(x["price"])) for x in order_summary["drinks"] + order_summary["sides"]]
assert prices == [Decimal("9.25"), Decimal("5.00"), Decimal("3.50")]
assert sum(prices) == Decimal(str(order_summary["total_price"])) == Decimal("17.75")
assert dashboard.get_root(Document()) is not None
other, isolated, _, _, _ = build_pizzabot(pizzabot_system)
assert isolated is not context and len(isolated) == 1
before = context.copy()
inp.value_input = "   "
button_conversation.clicks += 1
assert context == before and not button_conversation.disabled
assert chat_turn(context, "") is None and context == before
assert "&lt;script&gt;" in html.escape("<script>")
assert "Total: $17.75" in context[-1]["content"]
for item, sizes in MENU.items():
    for size, price in sizes.items():
        text, total = render_order_response({"reply": "Confirmed", "items": [{"item": item, "size": size, "quantity": 2}]})
        assert total == Decimal(price) * 2
for bad in [{"reply": "Total: $19.25", "items": []}, {"reply": "Ok", "items": [{"item": "unknown", "size": "regular", "quantity": 1}]}, {"reply": "Ok", "items": [{"item": "cheese pizza", "size": "regular", "quantity": 1}]}, {"reply": "Ok", "items": [{"item": "fries", "size": "regular", "quantity": True}]}, {"reply": "Ok", "items": [{"item": "fries", "size": "regular", "quantity": 0}]}]:
    try: render_order_response(bad)
    except ValueError: pass
    else: raise AssertionError("Invalid model price/item accepted")
print("PASS: real role/history examples; four registered UI callbacks; Python-calculated 9.25 + 5.00 + 3.50 = 17.75 receipt; every menu price/size and invalid-price/item/quantity cases; isolated state, empty-input no-op, escaped display and Bokeh document construction. No real order or payment.")

## Original source appendix (not executed)

Historical code and transcripts are retained verbatim below, not instructions to run obsolete APIs or paste credentials.

### Original pre block 0

````text
pip install openai
````

### Original pre block 1

````text
import openai
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key  = os.getenv('OPENAI_API_KEY')
````

### Original pre block 2

````text
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message["content"]
````

### Original pre block 3

````text
def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
#     print(str(response.choices[0].message))
    return response.choices[0].message["content"]
````

### Original pre block 4

````text
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]
````

### Original pre block 5

````text
response = get_completion_from_messages(messages, temperature=1)
print(response)
````

### Original pre block 6

````text
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Youssef'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)
````

### Original pre block 7

````text
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)
````

### Original pre block 8

````text
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Youssef'},
{'role':'assistant', 'content': "Hi Youssef! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)
````

### Original pre block 9

````text
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context) 
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels)
````

### Original pre block 10

````text
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard
````

### Original pre block 11

````text
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)
````

### Original transcript 0

````text
To get to the other side, my dear interlocutor!
````

### Original transcript 1

````text
Hello Youssef! It’s nice to meet you. How are you doing today?
````

### Original transcript 2

````text
I’m sorry, but I am not able to remember information like your name. How about you tell me your name so I can remember it this time?
````

### Original transcript 3

````text
Your name is Youssef. If you need any more reminders, feel free to ask!
````

### Original transcript 4

````text
{
 “pizza”: {
 “type”: “cheese”,
 “size”: “medium”
 },
 “toppings”: [],
 “drinks”: [
 {
 “type”: “bottled water”,
 “size”: “regular”
 }
 ],
 “sides”: [
 {
 “type”: “fries”,
 “size”: “regular”
 }
 ],
 “total price”: 17.75
}
````